Calculate and save long-term biomass change of each biological functional group as a result of PAH toxicity from dilbit spill scenarios at Turn Point/Haro Strait

In [1]:
import os
import xarray as xr
import numpy as np
import itertools
import pandas as pd
from pathlib import Path
import geopandas as gpd
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import ssam_groups as groups

### Define scenario and control files

| Scenarios	|	Description |
|-----------|---------------|
| 1	|	no management |
| 2 |   30-day fisheries closures |
| 3	|	90-day fisheries closures |
| 4	|	spill containment within 48hrs |
| 5	|	spill containment within 48hrs + 30-day fisheries closures |
| 6	|	spill containment within 48hrs + 90-day fisheries closures |

8 simulations for each scenario
1. winter S winds
1. winter N winds
1. spring opposing winds & currents
1. spring tandem winds & currents
1. summer Fraser + strong winds
1. summer Fraser + weak winds
1. fall strong N winds
1. fall weak winds

In [2]:
# Read in salish sea atlantis output files.
control_file = "/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/control-2039-2042/outputSalishSea.nc"
control = xr.open_dataset(str(control_file), decode_cf=True)
time = np.ma.filled(control.variables['t'])

In [3]:
scenario_root = Path('/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/')
scenario_paths = sorted([p for p in scenario_root.glob('highres-2039-2042_5b*/outputSalishSea.nc')])
for path in scenario_paths:
    print(path.parent.stem)

highres-2039-2042_5b_1_2019-01-20
highres-2039-2042_5b_1_2019-04-12
highres-2039-2042_5b_1_2019-07-03
highres-2039-2042_5b_1_2019-10-20
highres-2039-2042_5b_1_2020-01-24
highres-2039-2042_5b_1_2020-04-11
highres-2039-2042_5b_1_2020-07-05
highres-2039-2042_5b_1_2020-10-20
highres-2039-2042_5b_2_2019-01-20
highres-2039-2042_5b_2_2019-04-12
highres-2039-2042_5b_2_2019-07-03
highres-2039-2042_5b_2_2019-10-20
highres-2039-2042_5b_2_2020-01-24
highres-2039-2042_5b_2_2020-04-11
highres-2039-2042_5b_2_2020-07-05
highres-2039-2042_5b_2_2020-10-20
highres-2039-2042_5b_3_2019-01-20
highres-2039-2042_5b_3_2019-04-12
highres-2039-2042_5b_3_2019-07-03
highres-2039-2042_5b_3_2019-10-20
highres-2039-2042_5b_3_2020-01-24
highres-2039-2042_5b_3_2020-04-11
highres-2039-2042_5b_3_2020-07-05
highres-2039-2042_5b_3_2020-10-20
highres-2039-2042_5b_4_2019-01-20
highres-2039-2042_5b_4_2019-04-12
highres-2039-2042_5b_4_2019-07-03
highres-2039-2042_5b_4_2019-10-20
highres-2039-2042_5b_4_2020-01-24
highres-2039-2

In [4]:
scenario_datasets = [xr.open_dataset(scen,decode_cf=True) for scen in scenario_paths]

In [ ]:
# time after burn-in
start = 0
end = time.size-1

### Calculate Mean of Final 3 years (2039-2042)

In [ ]:
def mean_data_weightatage(bio_group, location=groups.salish_sea):
    results = []

    for scenario, path in zip(scenario_datasets, scenario_paths):
        nm = str(path.parent.stem).split(sep='_')
        scenario_name = nm[2]
        scenario_description = groups.scenarios[nm[2]]
        simulation = nm[3]
        simulation_description = groups.conditions[nm[3]]

        for species in bio_group:
            numCohorts = groups.cohorts[bio_group[species]]
            
            for cohort in range (1, numCohorts+1):

                new_species = bio_group[species] + str(cohort)
            
                o_numbers_tbl = np.ma.filled(scenario.variables[new_species + '_Nums'][start:end, location, 0:6], np.nan)
                o_structuralN_tbl = np.ma.filled(scenario.variables[new_species +'_StructN'][start:end, location, 0:6], np.nan)
                o_reservedN_tbl = np.ma.filled(scenario.variables[new_species +'_ResN'][start:end, location, 0:6], np.nan)

                c_numbers_tbl = np.ma.filled(control.variables[new_species + '_Nums'][start:end, location, 0:6], np.nan)
                c_structuralN_tbl = np.ma.filled(control.variables[new_species +'_StructN'][start:end, location, 0:6], np.nan)
                c_reservedN_tbl = np.ma.filled(control.variables[new_species +'_ResN'][start:end, location, 0:6], np.nan)

                o_weightatage_tbl = (o_structuralN_tbl + o_reservedN_tbl) * o_numbers_tbl 
                o_weightatage = o_weightatage_tbl.sum(axis=(1,2)).mean()

                c_weightatage_tbl = (c_structuralN_tbl + c_reservedN_tbl) * c_numbers_tbl 
                c_weightatage = c_weightatage_tbl.sum(axis=(1,2)).mean()

                ratio = (o_weightatage / c_weightatage - 1) * 100
            
                results.append({
                    'bio_group': species,
                    'cohort':cohort
                    'scenario': scenario_name,
                    'description': scenario_description,
                    'simulation': simulation,
                    'sim_description': simulation_description,
                    'percent_change': ratio,
                })

    df = pd.DataFrame(results)
    df.to_csv("/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/CohortScen"+scenario_name+"_"+bio_group[species]+"_weightatage.csv")

In [ ]:
def mean_data_biomass(bio_group, location=groups.salish_sea):
    results = []

    for scenario, path in zip(scenario_datasets, scenario_paths):
        nm = str(path.parent.stem).split(sep='_')
        scenario_name = nm[2]
        scenario_description = groups.scenarios[nm[2]]
        simulation = nm[3]
        simulation_description = groups.conditions[nm[3]]

        for species in bio_group:
            numCohorts = groups.cohorts[bio_group[species]]

            for cohort in range (1, numCohorts+1):

                new_species = bio_group[species] + str(cohort)
            
                o_structuralN_tbl = np.ma.filled(scenario.variables[new_species +'_StructN'][start:end, location, 0:6], np.nan)
                o_reservedN_tbl = np.ma.filled(scenario.variables[new_species +'_ResN'][start:end, location, 0:6], np.nan)

                c_structuralN_tbl = np.ma.filled(control.variables[new_species +'_StructN'][start:end, location, 0:6], np.nan)
                c_reservedN_tbl = np.ma.filled(control.variables[new_species +'_ResN'][start:end, location, 0:6], np.nan)

                o_biomass_tbl = (o_structuralN_tbl + o_reservedN_tbl)
                o_biomass = o_biomass + o_biomass_tbl.sum(axis=(1,2)).mean()

                c_biomass_tbl = (c_structuralN_tbl + c_reservedN_tbl) 
                c_biomass = c_biomass + c_biomass_tbl.sum(axis=(1,2)).mean()

                ratio = (o_biomass / c_biomass - 1) * 100
            
                results.append({
                    'bio_group': species,
                    'cohort':cohort
                    'scenario': scenario_name,
                    'description': scenario_description,
                    'simulation': simulation,
                    'sim_description': simulation_description,
                    'percent_change': ratio,
                })

    df = pd.DataFrame(results)
    df.to_csv("/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/CohortScen"+scenario_name+"_"+bio_group[species]+"_biomass.csv")

In [ ]:
def mean_data_numbers(bio_group, location=groups.salish_sea):
    results = []

    for scenario, path in zip(scenario_datasets, scenario_paths):
        nm = str(path.parent.stem).split(sep='_')
        scenario_name = nm[2]
        scenario_description = groups.scenarios[nm[2]]
        simulation = nm[3]
        simulation_description = groups.conditions[nm[3]]

        for species in bio_group:
            numCohorts = groups.cohorts[bio_group[species]]

            for cohort in range (1, numCohorts+1):

                new_species = bio_group[species] + str(cohort)
            
                o_numbers_tbl = np.ma.filled(scenario.variables[new_species + '_Nums'][start:end, location, 0:6], np.nan)
                c_numbers_tbl = np.ma.filled(control.variables[new_species + '_Nums'][start:end, location, 0:6], np.nan)
 
                
                o_numbers = o_numbers + o_numbers_tbl.sum(axis=(1,2)).mean()
                c_numbers = c_numbers + c_numbers_tbl.sum(axis=(1,2)).mean()

                ratio = (o_numbers / c_numbers - 1) * 100
            
                results.append({
                    'bio_group': species,
                    'cohort':cohort
                    'scenario': scenario_name,
                    'description': scenario_description,
                    'simulation': simulation,
                    'sim_description': simulation_description,
                    'percent_change': ratio,
                })

    df = pd.DataFrame(results)
    df.to_csv("/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/CohortScen"+scenario_name+"_"+bio_group[species]+"_numbers.csv")

In [ ]:
mean_data_numbers(groups.salmon)

In [ ]:
mean_data_numbers(groups.named_fish)

In [ ]:
mean_data_numbers(groups.other_fish)

In [ ]:
mean_data_numbers(groups.mammals)

In [ ]:
mean_data_numbers(groups.sharks)

In [ ]:
mean_data_numbers(groups.birds)

In [ ]:
results_root = Path('/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/')
results_paths = sorted([p for p in results_root.glob('CohortScen*.csv')])
mean_data = []
for file in results_paths:
    df1 = pd.read_csv(file)
    mean_data.append(df1[['bio_group', 'cohort','scenario', 'description','simulation','sim_description','percent_change',]])

mean_data_df = pd.concat(mean_data, ignore_index=True)
mean_data_df.to_csv("/ocean/rlovindeer/MOAD/analysis-raisha/SSmodel_outputs/Spills/MPRI2/cohort-numbers.csv")